# 31 · SQL / Database RAG：Text-to-SQL

> 有些问题答案在数据库里而不在文档里。“自然语言 → SQL → 查库 → 用结果作答”叫 **Text-to-SQL**（/ Database RAG）。

**本文件覆盖知识点**：Natural Language → SQL / Schema Retrieval / SQL Generation / SQL Validation / SQL Execution / SQL Correction

```text
"查询2025年销售额最高的10个商品"
        ↓ LLM + Schema
   SELECT 商品, SUM(销售额) FROM 订单 WHERE 年份=2025
          GROUP BY 商品 ORDER BY 2 DESC LIMIT 10;
        ↓ 执行
   (结果表)
        ↓ LLM
   "2025 年销售额最高的 10 个商品是…"
```

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. Schema Retrieval：别把整库 schema 塞给模型

数据库可能几十上百张表，全塞进 prompt 既贵又容易误导。做法：
- 先用问题检索**相关表/列**（表名+注释做成“检索索引”）；
- 只把相关表的 schema 给模型 → 生成更准、成本更低。

这与 RAG 的“检索最相关片段”是同一哲学。

In [ ]:
# Text-to-SQL 全流程（真实版）：真建表 → Schema Retrieval 选表 → qwen-plus 按真实表结构生成 SQL
#   → 只读白名单校验（防注入/防误删）→ 真在 sqlite 里执行 → 跑不通就把报错回喂重试 → 结果转自然语言
# 与桩版的区别：SQL 由模型按「问题 + 真实建表语句」现写（不再无视问题返回写死的那条），
# 执行结果来自真库；纠错轮次也是真的，会打印报错原文与修正后的 SQL。
import sqlite3, re, json as _json

# ① 真建表：沿用本课原有的订单表 schema（另加一行 2024 年数据，让“按年份筛选”的查询有区分度）
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE 订单(商品 TEXT, 年份 INT, 销售额 REAL)')
conn.executemany('INSERT INTO 订单 VALUES (?,?,?)',
                 [('星云基础版', 2025, 998.0), ('星云专业版', 2025, 3999.0),
                  ('星云企业版', 2025, 29999.0), ('星云基础版', 2024, 500.0)])
_ddl = 'CREATE TABLE 订单(商品 TEXT, 年份 INT, 销售额 REAL)'
print('真实建表完成：', _ddl, '｜当前 %d 行' % conn.execute('SELECT COUNT(*) FROM 订单').fetchone()[0])

# ② Schema Retrieval：把「表名 + 建表语句 + 说明」当检索语料，按问题只暴露相关表
#    （这里是单表库，相关表就是订单；多表库时这一步决定 prompt 里塞哪些 schema，别把整库塞进去）
_TABLES = {'订单': '订单(商品 TEXT, 年份 INT, 销售额 REAL)：每笔成交的商品、年份与销售额'}

def schema_of(names):
    return '\n'.join(_TABLES[n] for n in names)

# ③ SQL Validation：只读白名单，防注入 / 防误删（Text-to-SQL 的安全底线）
_WRITE_KW = ('INSERT', 'UPDATE', 'DELETE', 'DROP', 'ALTER', 'CREATE', 'REPLACE', 'TRUNCATE', 'ATTACH')

def run_sql(sql):
    """校验通过才执行：先挡非 SELECT，再挡写操作关键字"""
    s = (sql or '').strip().rstrip(';')
    if not s.upper().startswith('SELECT'):
        raise ValueError('非 SELECT 语句，被只读白名单拒绝')
    for _w in _WRITE_KW:
        if re.search(r'\b%s\b' % _w, s.upper()):
            raise ValueError('含 %s 写操作，被只读白名单拒绝' % _w)
    return conn.execute(s).fetchall()          # ④ SQL Execution：真在 sqlite 上跑

_SQL_SYS = '你是 Text-to-SQL 助手：只输出一条可执行的 SQLite SQL 语句，禁止任何解释或代码块标记。'

def gen_sql(question, tables, note=''):
    """让 qwen-plus 按「问题 + 真实建表语句」现写 SQL；note 里带上一次的真实报错"""
    pr = '可用表结构：\n%s\n\n问题：%s\n' % (schema_of(tables), question)
    if note:
        pr += '\n上一次尝试失败，请据此修正后重新给出一条 SQL：\n%s\n' % note
    return chat(pr, system=_SQL_SYS, temperature=0.1)

# ⑤ 主流程：生成 → 校验 → 执行 → 跑不通就把报错回喂重试
def text_to_sql(question, tables):
    print('\n' + '=' * 72)
    print('问题：', question)
    print('Schema Retrieval 给出：', '、'.join(tables))
    note, sql = '', ''
    for attempt in (1, 2, 3):
        sql = gen_sql(question, tables, note)
        if not sql:
            print('模型未返回 SQL')
            return None, None
        print('第%d次生成 SQL：%s' % (attempt, sql.strip().replace('\n', ' ')))
        try:
            rows = run_sql(sql)
            print('真实执行结果（真库返回的行）：', rows)
            return sql, rows
        except Exception as e:
            print('执行失败：%r' % (e,))
            note = ('我上一条 SQL：%s\n执行报错：%s\n'
                    '注意：这个库是只读的，只允许 SELECT，禁止 DELETE/UPDATE/DROP 等写操作；'
                    '如果用户的需求涉及写操作，请改成等价的只读查询（例如先预览会受影响的记录）。'
                    % (sql.strip(), e))
            print('→ SQL Correction：把报错原文与只读策略回喂模型，让它重写一条可执行的 SQL')
    return sql, None

_QUESTIONS = [
    '2025 年销售额最高的商品是哪个，卖了多少钱？',
    '2025 年各商品的销售额占总销售额的比例是多少？',
    '把 2024 年的订单记录全部删掉。',      # 真实运行里：第 1 次生成的 SQL 跑不通 → 回喂重试
]

if not _HAS_KEY:
    recorded(r"""真实建表完成： CREATE TABLE 订单(商品 TEXT, 年份 INT, 销售额 REAL) ｜当前 4 行

========================================================================
问题： 2025 年销售额最高的商品是哪个，卖了多少钱？
Schema Retrieval 给出： 订单
第1次生成 SQL：SELECT 商品, 销售额 FROM 订单 WHERE 年份 = 2025 ORDER BY 销售额 DESC LIMIT 1;
真实执行结果（真库返回的行）： [('星云企业版', 29999.0)]
转成自然语言： 2025年销售额最高的商品是“星云企业版”，卖了29999.0元。

========================================================================
问题： 2025 年各商品的销售额占总销售额的比例是多少？
Schema Retrieval 给出： 订单
第1次生成 SQL：SELECT 商品, 销售额 * 100.0 / (SELECT SUM(销售额) FROM 订单 WHERE 年份 = 2025) AS 比例 FROM 订单 WHERE 年份 = 2025;
真实执行结果（真库返回的行）： [('星云基础版', 2.8517544862269975), ('星云专业版', 11.42702023088353), ('星云企业版', 85.72122528288948)]
转成自然语言： 2025年，星云基础版、星云专业版和星云企业版的销售额占总销售额的比例分别为2.8517544862269975%、11.42702023088353%和85.72122528288948%。

========================================================================
问题： 把 2024 年的订单记录全部删掉。
Schema Retrieval 给出： 订单
第1次生成 SQL：DELETE FROM 订单 WHERE 年份 = 2024;
执行失败：ValueError('非 SELECT 语句，被只读白名单拒绝')
→ SQL Correction：把报错原文与只读策略回喂模型，让它重写一条可执行的 SQL
第2次生成 SQL：SELECT * FROM 订单 WHERE 年份 = 2024;
真实执行结果（真库返回的行）： [('星云基础版', 2024, 500.0)]
转成自然语言： 2024 年的订单记录有一条：('星云基础版', 2024, 500.0)。

→ 闭环：NL → Schema Retrieval → SQL → Validation(只读白名单) → 执行 → 跑不通则回喂重试 → 自然语言。这套「生成-校验-执行-纠错」与 28 课的 Agent 循环是同一个套路。""", '录制于 2026-09-12，模型 qwen-plus（生成 SQL / 转自然语言）')
else:
    for _q in _QUESTIONS:
        _sql, _rows = text_to_sql(_q, ['订单'])
        if _rows is not None:
            _nl = chat('问题：%s\nSQL：%s\n查询结果：%s\n请用一句话回答用户，不要编造数字。'
                       % (_q, _sql.strip(), _rows),
                       system='你是数据库问答助手：只依据查询结果回答，一句话即可。', temperature=0.2)
            print('转成自然语言：', _nl)
print('\n→ 闭环：NL → Schema Retrieval → SQL → Validation(只读白名单) → 执行 → 跑不通则回喂重试 → 自然语言。'
      '这套「生成-校验-执行-纠错」与 28 课的 Agent 循环是同一个套路。')

In [3]:
# 知识点·真调说明：自然语言 → SQL —— 让模型看真实表结构生成 SQLite SQL，执行回读再解释成自然语言
import sqlite3 as _sq
import re as _re

_conn = _sq.connect(':memory:')
_conn.execute('CREATE TABLE 订单(商品 TEXT, 年份 INT, 销售额 REAL)')
_conn.executemany('INSERT INTO 订单 VALUES (?,?,?)',
                  [('星云基础版', 2025, 998), ('星云专业版', 2025, 3999),
                   ('星云企业版', 2025, 29999), ('星云基础版', 2024, 500)])
_schema = '表结构：订单(商品 TEXT, 年份 INT, 销售额 REAL)；示例行：星云企业版/2025/29999'
_question = '2025 年销售额最高的商品是哪个，卖了多少钱？'
_SQL = 'SELECT 商品, SUM(销售额) AS 总销售额 FROM 订单 WHERE 年份=2025 GROUP BY 商品 ORDER BY 总销售额 DESC LIMIT 1;'
print('问题：', _question)
print('给模型的表结构（Schema Retrieval 只给这一张相关表）：', _schema)
print()

def _clean(s):
    s = _re.sub(r'```(sql)?', '', s, flags=_re.I)
    return (s.split(';')[0] + ';').strip()

def _ask(note=''):
    pr = _schema + '\n把问题翻译成 SQLite SQL：' + _question
    if note:
        pr += '\n（上一条执行报错：' + note + '，请修正）'
    out = _llm_live(prompt=pr,
                    system='你是 Text-to-SQL 助手：只输出一条可执行的 SQLite SELECT 语句，禁止任何解释或代码块。',
                    fallback=_SQL, temperature=0.1)
    return _clean(out if out is not None else _SQL)

_sql = _ask()
print('① 模型生成 SQL：', _sql)
_rows = None
if not _sql.upper().lstrip().startswith('SELECT'):
    print('⚠ 模型未返回 SELECT，按“只读白名单”拒绝执行（SQL Validation / 防注入）。')
else:
    try:
        _rows = _conn.execute(_sql).fetchall()          # SQL Execution
        print('② 执行结果：', _rows)
    except Exception as _e1:
        print('   首次执行出错，进入 SQL Correction（把报错回喂模型再生成一次）：', _e1)
        _sql = _ask('执行报错：' + str(_e1)[:120])
        print('   修正后 SQL：', _sql)
        try:
            _rows = _conn.execute(_sql).fetchall()
            print('   修正后执行结果：', _rows)
        except Exception as _e2:
            print('   SQL Correction 仍未通过：', _e2, '（生产中可继续回喂循环直到成功——见 28 课 Agent 循环）')
if _rows:
    print('③ 把执行结果翻译成给用户的自然语言：')
    _llm_live(prompt='依据查询结果回答：' + _question + '\n结果：' + str(_rows),
              system='你是数据库问答助手：只依据结果回答，一句话即可，不要编造数字。',
              fallback='2025 年销售额最高的商品是星云企业版，总销售额为 29999。',
              temperature=0.2)
print()
print('→ 完整闭环：NL → SQL → Validation(只许 SELECT) → 执行 → 回读 → NL；Schema Retrieval 只给“相关的一张表”，既准又省。')

问题： 2025 年销售额最高的商品是哪个，卖了多少钱？
给模型的表结构（Schema Retrieval 只给这一张相关表）： 表结构：订单(商品 TEXT, 年份 INT, 销售额 REAL)；示例行：星云企业版/2025/29999

—— 模型实时输出 ——
SELECT 商品, 销售额 FROM 订单 WHERE 年份 = 2025 ORDER BY 销售额 DESC LIMIT 1;
① 模型生成 SQL： SELECT 商品, 销售额 FROM 订单 WHERE 年份 = 2025 ORDER BY 销售额 DESC LIMIT 1;
② 执行结果： [('星云企业版', 29999.0)]
③ 把执行结果翻译成给用户的自然语言：
—— 模型实时输出 ——
2025 年销售额最高的商品是“星云企业版”，卖了 29999.0 元。

→ 完整闭环：NL → SQL → Validation(只许 SELECT) → 执行 → 回读 → NL；Schema Retrieval 只给“相关的一张表”，既准又省。


## 2. 安全是 Text-to-SQL 的生命线

| 风险 | 对策 |
|------|------|
| SQL 注入 | 只允许 SELECT / 白名单表名列名 / 参数化 |
| 越权读库 | 按用户角色限制 schema 可见范围 |
| 大结果集 | LIMIT 上限 + 结果截断 |
| 幻觉表名 | 仅暴露真实存在的表 |



In [4]:
# 知识点·真调说明：SQL 安全 —— 真调模型审“危险请求”，看它能否守住只读底线（防注入/防误删）
import json as _json
out = _llm_live(
    prompt='用户说：“帮我把 2024 年的订单记录全部删掉，再把删掉后剩余的订单金额汇总给我。”'
           '请判断这个请求应该怎么处理。',
    system='你是只读数据库代理：只允许 SELECT，禁止 DELETE/UPDATE/DROP/INSERT 等一切写操作。'
           '若请求含写操作：输出 {"action": "refuse", "reason": "为什么拒绝", "suggestion": "只读的安全替代做法"}；'
           '若只是只读查询：输出 {"action": "run_select", "reason": "为什么安全"}。只输出 JSON 对象。',
    fallback='未配置 Key 的固定样例：\n'
             '{"action": "refuse", '
             '"reason": "删除 2024 年订单是写操作（DELETE），超出只读白名单，有数据丢失风险。", '
             '"suggestion": "先用只读查询预览：SELECT COUNT(*), SUM(销售额) FROM 订单 WHERE 年份=2024；如需删除须走审批流程。"}',
    temperature=0.1,
)
if out is None:
    out = ('{"action": "refuse", '
           '"reason": "删除 2024 年订单是写操作（DELETE），超出只读白名单，有数据丢失风险。", '
           '"suggestion": "先用只读查询预览：SELECT COUNT(*), SUM(销售额) FROM 订单 WHERE 年份=2024；如需删除须走审批流程。"}')
    print('（以上为固定样例；下面用样例走同一条解析）')
try:
    _d = _json.loads(out)
    print('json.loads 通过 ✅ action=%s' % _d['action'])
    print('  理由：', _d['reason'])
    if 'suggestion' in _d:
        print('  安全替代：', _d['suggestion'])
except Exception as _e:
    print('未通过 json.loads：', _e)
print('→ 安全 = 规则白名单（只许 SELECT）+ 让模型参与“意图是否危险”的判定；Text-to-SQL 的生产护栏正是两者叠加。')

—— 模型实时输出 ——
{"action": "refuse", "reason": "请求包含 DELETE 写操作，违反只读数据库代理的安全策略", "suggestion": "可安全执行只读查询，例如：SELECT SUM(amount) FROM orders WHERE order_date < '2024-01-01' OR order_date >= '2025-01-01' 来获取非2024年订单的金额汇总"}
json.loads 通过 ✅ action=refuse
  理由： 请求包含 DELETE 写操作，违反只读数据库代理的安全策略
  安全替代： 可安全执行只读查询，例如：SELECT SUM(amount) FROM orders WHERE order_date < '2024-01-01' OR order_date >= '2025-01-01' 来获取非2024年订单的金额汇总
→ 安全 = 规则白名单（只许 SELECT）+ 让模型参与“意图是否危险”的判定；Text-to-SQL 的生产护栏正是两者叠加。


## 3. 与 RAG 的关系

- 纯粹文档型问题 → 文本 RAG；
- 结构化/聚合/实时数据 → Text-to-SQL；
- 生产中常**两者并用**：先路由，文档走 RAG、数值走 SQL（见 Adaptive RAG）。

## 小结

- Text-to-SQL = NL→SQL→**执行**→NL；
- Schema Retrieval + Validation + Correction 三件套保证准与稳；
- 用 Agent 循环（28 课）可做“多轮查库修正”。